In [5]:
# ----------------- Step 0: Install Required Packages -----------------
!pip install -q pandas pyarrow sentence-transformers faiss-cpu chromadb openai


[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_parquet(r"C:\Users\manjo\Downloads\wiki_hybrid_chunks_300 (1).parquet")
print("Available columns:\n", df.columns)

Available columns:
 Index(['group_id', 'section', 'chunk_id', 'chunk_text'], dtype='object')


In [3]:
# Use the correct text column
df['text'] = df['chunk_text'].astype(str)
texts = df['text'].tolist()

In [4]:
# -------------------- Step 2: Generate Embeddings --------------------
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(texts, show_progress_bar=True)


Batches:   0%|          | 0/314 [00:00<?, ?it/s]

In [7]:
# -------------------- Step 3: Save Embeddings to New Parquet --------------------
import numpy as np

embedding_df = pd.DataFrame(embeddings, columns=[f"emb_{i}" for i in range(len(embeddings[0]))])
df_with_embeddings = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)
df_with_embeddings.to_parquet("wiki_chunks_with_embeddings.parquet")
print("✅ Saved: wiki_chunks_with_embeddings.parquet")

✅ Saved: wiki_chunks_with_embeddings.parquet


In [8]:
# -------------------- Step 4A: Store in FAISS --------------------
import faiss

embedding_array = np.array(embeddings).astype("float32")
faiss_index = faiss.IndexFlatL2(embedding_array.shape[1])
faiss_index.add(embedding_array)
faiss.write_index(faiss_index, "wiki_chunks_faiss.index")
print("✅ Saved FAISS index")

✅ Saved FAISS index


In [9]:
# -------------------- Step 5: Retrieve Relevant Chunks --------------------
query = "What is the use of reinforcement learning in chatbots?"
query_embedding = model.encode([query]).astype("float32")

D, I = faiss_index.search(query_embedding, k=3)
retrieved_chunks = [texts[i] for i in I[0]]
retrieved_context = " ".join(retrieved_chunks)

In [ ]:
# Generate Answer Using Local LLM

In [10]:
pip install transformers accelerate torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#load a local LLM (Lightweight Example)

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

In [ ]:
#Build prompt from Retrieved Chunks

In [12]:
def build_prompt(context, query):
    return f"""You are an intelligent assistant. Use the following context to answer the question.

Context:
{context}

Question: {query}
Answer:"""

In [ ]:
#Generate Answer from LLM

In [17]:
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cpu")
    output = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [18]:
from random import choice
from diskcache import Cache

# Initialize cache
cache = Cache("./llm_cache")

# Define fallback logic
fallback_responses = [
    "I'm sorry, I couldn't find a helpful answer. Could you rephrase your question?",
    "Hmm... I didn’t quite get that. Can you ask it in a different way?",
    "I’m still learning! Can you try asking it another way?",
    "Apologies, I’m unable to help with that one. Could you provide more context?"
]

generic_phrases = [
    "you are an intelligent assistant",
    "i'm sorry",
    "i don't know",
    "as an ai"
]

# Step: Combine Everything with Fallback + Cache
query = "What is the use of reinforcement learning in chatbots?"

# Check cache
if query in cache:
    answer = cache[query]
    print("✅ Answer retrieved from cache.")
else:
    # Get relevant chunks from FAISS
    query_embedding = model.encode([query]).astype("float32")
    D, I = faiss_index.search(query_embedding, k=3)
    retrieved_chunks = [texts[i] for i in I[0]]
    retrieved_context = "\n".join(retrieved_chunks)

    # Build prompt and get answer
    prompt = build_prompt(retrieved_context, query)
    answer = generate_answer(prompt)

    # Apply fallback logic if response is weak
    if not answer.strip() or any(phrase in answer.lower() for phrase in generic_phrases):
        print("⚠️ Fallback triggered.")
        fallback = choice(fallback_responses)
        answer = fallback + "\nTip: Try being more specific, like mentioning a topic or keyword."

    # Save to cache
    cache[query] = answer

# Output
print("\n🧠 Final Answer:\n", answer)

✅ Answer retrieved from cache.

🧠 Final Answer:
 I'm sorry, I couldn't find a helpful answer. Could you rephrase your question?
Tip: Try being more specific, like mentioning a topic or key phrase.
